In [1]:
import pandas as pd
import requests
from typing import Dict, List
import re

In [2]:
def fetch_users(results: int = 10):
    url = "https://randomuser.me/api/"
    params = {"results": results}

    try:
        response = requests.get(url, params=params, timeout=10)
        response.raise_for_status()
        return response.json()

    except requests.exceptions.RequestException as e:
        print(f"[Error] API request failed: {e}")
        return {"results": []}

def transform_users(data: Dict):
    users = data.get("results", [])
    records = []

    for user in users:
        record = {
            "full_name": f"{user.get('name', {}).get('first')} {user.get('name', {}).get('last')}",
            "age": user.get('dob', {}).get('age'),
            "gender": user.get('gender'),
            "email": user.get('email'),
            "phone": user.get('phone'),
            "city": user.get('location', {}).get('city'),
            "state": user.get('location', {}).get('state'),
            "country": user.get('location', {}).get('country'),
            "signup_date": user.get('registered', {}).get('date')            
        }
        records.append(record)

    return records

def build_dataframe(n_users: int=10):
    raw_data = fetch_users(n_users)
    records = transform_users(raw_data)
    df = pd.DataFrame(records)

    return df

if __name__ == "__main__":
    df = build_dataframe(15)

    print("\nSaas Users Dataset:\n")
    print(df.head())


Saas Users Dataset:

        full_name  age  gender                       email           phone  \
0  Albert Danchuk   69    male  albert.danchuk@example.com  (099) M70-0678   
1     Nick Turner   74    male     nick.turner@example.com    031-377-4957   
2   Dora Richards   75  female   dora.richards@example.com    07-1271-2868   
3  Ricardo Lozano   34    male  ricardo.lozano@example.com     977-772-700   
4     Ronnie Gray   26    male     ronnie.gray@example.com    016973 08579   

         city             state         country               signup_date  
0      Kiliya           Donecka         Ukraine  2016-01-26T07:19:32.995Z  
1    Donabate         Waterford         Ireland  2007-01-06T15:30:42.499Z  
2  Townsville   South Australia       Australia  2006-07-03T03:09:38.215Z  
3    Torrente  Región de Murcia           Spain  2015-06-26T23:06:08.478Z  
4   Lichfield   Buckinghamshire  United Kingdom  2004-05-26T01:27:14.466Z  


In [19]:
# Remove all non-digit characters
df['phone_clean'] = df['phone'].apply(lambda x: re.sub(r'\D', '', str(x)))

# Optional: format into a consistent pattern, e.g. (XXX) XXX-XXXX
df['phone_clean'] = df['phone_clean'].apply(
    lambda x: f"({x[:3]}) {x[3:6]}-{x[6:]}" if len(x) >= 10 else x
)

# Remove milliseconds and Z
df['signup_date_clean'] = df['signup_date'].apply(
    lambda x: re.sub(r'\.\d+Z$', '', str(x))
)

# Convert to pandas datetime for analysis
df['signup_date_clean'] = pd.to_datetime(df['signup_date_clean'])

In [26]:
# Drop the columns
df.drop(columns=['phone', 'signup_date'], inplace=True)

# Rename columns
df.rename(columns={'phone_clean': 'phone', 'signup_date_clean': 'signup_date'}, inplace=True)

In [27]:
df

,full_name,age,gender,email,city,state,country,phone,signup_date
0,Albert Danchuk,69,male,albert.danchuk@example.com,Kiliya,Donecka,Ukraine,099700678,2016-01-26 07:19:32
1,Nick Turner,74,male,nick.turner@example.com,Donabate,Waterford,Ireland,(031) 377-4957,2007-01-06 15:30:42
2,Dora Richards,75,female,dora.richards@example.com,Townsville,South Australia,Australia,(071) 271-2868,2006-07-03 03:09:38
3,Ricardo Lozano,34,male,ricardo.lozano@example.com,Torrente,Región de Murcia,Spain,977772700,2015-06-26 23:06:08
4,Ronnie Gray,26,male,ronnie.gray@example.com,Lichfield,Buckinghamshire,United Kingdom,(016) 973-08579,2004-05-26 01:27:14
5,Gabe Mason,77,male,gabe.mason@example.com,Grand Prairie,Texas,United States,(755) 420-8816,2006-05-24 16:42:58
6,Mana Reimink,33,female,mana.reimink@example.com,Tuk,Groningen,Netherlands,(004) 689-2694,2021-07-17 00:34:51
7,Léandro Simon,39,male,leandro.simon@example.com,Pau,Calvados,France,(030) 767-9052,2021-05-01 15:40:32
8,Alicia Lavigne,30,female,alicia.lavigne@example.com,Campbellton,Saskatchewan,Canada,32059272,2004-09-12 20:27:57
9,Archer Green,55,male,archer.green@example.com,Timaru,Wellington,New Zealand,(132) 722-9479,2011-02-27 23:13:47


In [28]:
# save to csv file
df.to_csv("saas_users.csv", index=False)